In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

In [2]:
bonafide = pd.read_csv("../../Dataset/Features/bonafide_features.csv")
spoof = pd.read_csv("../../Dataset/Features/spoof_features.csv")
df = pd.concat([bonafide, spoof], ignore_index=True)

X = df.drop(['label', 'filename'], axis=1, errors='ignore').select_dtypes(include=[np.number])
y = df['label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=y, random_state=42)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train.values)
X_test_t = torch.FloatTensor(X_test)

In [3]:
class ProtoPNet(nn.Module):
    def __init__(self, input_dim, num_prototypes=10):
        super(ProtoPNet, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32)
        )
        self.prototypes = nn.Parameter(torch.randn(num_prototypes, 32))
        self.classifier = nn.Linear(num_prototypes, 2, bias=False)

    def forward(self, x):
        emb = self.encoder(x)
        distances = torch.cdist(emb, self.prototypes)
        similarity = torch.exp(-distances)
        logits = self.classifier(similarity)
        return logits, similarity

model = ProtoPNet(input_dim=29)

In [4]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
for epoch in range(50):
    logits, _ = model(X_train_t)
    loss = nn.CrossEntropyLoss()(logits, y_train_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [5]:
def explicar_con_prototipos(idx_muestra):
    muestra = X_test_t[idx_muestra].unsqueeze(0)
    logits, similarity = model(muestra)
    pred = torch.argmax(logits).item()
    
    # El prototipo que más "pesó" en la decisión
    proto_mas_cercano = torch.argmax(similarity).item()
    
    print(f"Predicción: {'REAL' if pred==1 else 'FAKE'}")
    print(f"Explicación: Esta muestra tiene su máxima similitud con el Prototipo ID: {proto_mas_cercano}")
    print(f"Valor de similitud: {similarity[0][proto_mas_cercano].item():.4f}")

explicar_con_prototipos(12)

Predicción: FAKE
Explicación: Esta muestra tiene su máxima similitud con el Prototipo ID: 1
Valor de similitud: 0.3559
